# From no DB to ORM

**DBPRA · TU Berlin**

We build the same warehouse / item application three times:

1. **Pure Python** — dicts in a list, persisted as JSON.
2. **Direct database access** — SQLite + `sqlite3`, schema with constraints, hand-written SQL.
3. **ORM** — SQLAlchemy: classes instead of CREATE TABLE, methods instead of SQL strings.

Each section runs the same operations and the same buggy inputs. Watch what changes — that is the lesson.

After Part 2 and Part 3 there is a **task** for you to fill in. By the end, decide for yourself: which version would you want to maintain?

## 1 — Pure Python


In [ ]:
def fresh_state():
    return {"warehouses": [], "items": [], "stores": [], "next_id": 1}

state = fresh_state()

def reset():
    global state
    state = fresh_state()

def _next_id():
    n = state["next_id"]
    state["next_id"] += 1
    return n

def add_warehouse(name, address):
    w = {"id": _next_id(), "name": name, "address": address}
    state["warehouses"].append(w)
    return w

def add_item(name, price, weight):
    i = {"id": _next_id(), "name": name, "price": price, "weight": weight}
    state["items"].append(i)
    return i

def stock(warehouse_id, item_id):
    state["stores"].append({"warehouse_id": warehouse_id, "item_id": item_id})

def items_at(warehouse_id):
    ids = {s["item_id"] for s in state["stores"] if s["warehouse_id"] == warehouse_id}
    return [i for i in state["items"] if i["id"] in ids]

reset()

Add some data and look at the state.


In [ ]:
add_warehouse("Berlin", "Alexanderplatz 1")
add_warehouse("Munich",  "Marienplatz 8")
add_item("Widget",  9.99, 0.5)
add_item("Gadget",  19.99, 1.2)
add_item("Sprocket", 4.50, 0.1)
stock(1, 1); stock(1, 2); stock(2, 2); stock(2, 3)

import json
print(json.dumps(state, indent=2))

### Trying to break it

Three things you would expect a real system to refuse:


In [ ]:
# 1. Two warehouses with the same name — accepted silently.
add_warehouse("Berlin", "another address")
print("warehouses named 'Berlin':",
      [w for w in state["warehouses"] if w["name"] == "Berlin"])

# 2. An item with a negative price — accepted silently.
add_item("Free money", -100, 0)
print("items with price < 0:",
      [i for i in state["items"] if i["price"] < 0])

# 3. Stocking at a warehouse that doesn't exist — accepted silently.
stock(warehouse_id=999, item_id=1)
print("stores rows referencing warehouse 999:",
      [s for s in state["stores"] if s["warehouse_id"] == 999])


### A query that should be easy

*"For each warehouse, show the total price of items stocked there, sorted by total descending."*

In pure Python this is a hand-written nested loop:


In [ ]:
def warehouse_totals():
    rows = []
    for w in state["warehouses"]:
        ids = {s["item_id"] for s in state["stores"] if s["warehouse_id"] == w["id"]}
        total = sum(i["price"] for i in state["items"] if i["id"] in ids)
        rows.append((w["name"], total))
    rows.sort(key=lambda r: -r[1])
    return rows

warehouse_totals()


**What we don't have:**
- **no persistence** — restart the kernel and everything is gone; we'd have to hand-roll JSON dumping and loading
- **no constraints** — duplicates, negative prices, dangling references all slipped through
- **no declarative queries** — we wrote a procedural loop for what should be one line
- no concurrent writers, no indexes, no transactions

These are the things a database gives us. Onward.

## 2 — Direct database access (SQLite + `sqlite3`)


In [ ]:
import sqlite3
from pathlib import Path

DB_FILE = Path("warehouse_dda.db")
if DB_FILE.exists():
    DB_FILE.unlink()

con = sqlite3.connect(DB_FILE)
con.execute("PRAGMA foreign_keys = ON")

con.executescript("""
CREATE TABLE warehouse (
  id      INTEGER PRIMARY KEY,
  name    TEXT NOT NULL UNIQUE,
  address TEXT NOT NULL
);
CREATE TABLE item (
  id     INTEGER PRIMARY KEY,
  name   TEXT NOT NULL,
  price  REAL NOT NULL CHECK (price  >= 0),
  weight REAL NOT NULL CHECK (weight >= 0)
);
CREATE TABLE stores (
  warehouse_id INTEGER NOT NULL REFERENCES warehouse(id),
  item_id      INTEGER NOT NULL REFERENCES item(id),
  PRIMARY KEY (warehouse_id, item_id)
);
""")
con.commit()

def add_warehouse(name, address):
    cur = con.execute("INSERT INTO warehouse (name, address) VALUES (?, ?)", (name, address))
    con.commit()
    return cur.lastrowid

def add_item(name, price, weight):
    cur = con.execute("INSERT INTO item (name, price, weight) VALUES (?, ?, ?)", (name, price, weight))
    con.commit()
    return cur.lastrowid

def stock(warehouse_id, item_id):
    con.execute("INSERT INTO stores (warehouse_id, item_id) VALUES (?, ?)", (warehouse_id, item_id))
    con.commit()

def items_at(warehouse_id):
    return con.execute(
        "SELECT i.id, i.name, i.price FROM item i "
        "JOIN stores s ON s.item_id = i.id WHERE s.warehouse_id = ?",
        (warehouse_id,),
    ).fetchall()

def warehouse_totals():
    return con.execute("""
        SELECT w.name, SUM(i.price) AS total
        FROM warehouse w
        JOIN stores s ON s.warehouse_id = w.id
        JOIN item   i ON i.id           = s.item_id
        GROUP BY w.id, w.name
        ORDER BY total DESC
    """).fetchall()

add_warehouse("Berlin", "Alexanderplatz 1")
add_warehouse("Munich",  "Marienplatz 8")
add_item("Widget",  9.99, 0.5)
add_item("Gadget",  19.99, 1.2)
add_item("Sprocket", 4.50, 0.1)
stock(1, 1); stock(1, 2); stock(2, 2); stock(2, 3)

warehouse_totals()


### The same three buggy inputs


In [ ]:
for label, action in [
    ("duplicate warehouse name", lambda: add_warehouse("Berlin", "x")),
    ("negative price",            lambda: add_item("Free money", -100, 0)),
    ("FK to missing warehouse",   lambda: stock(999, 1)),
]:
    try:
        action()
        print(f"{label}: NOT BLOCKED")
    except sqlite3.IntegrityError as e:
        print(f"{label}: blocked — {e}")


### Your task

Implement `warehouses_with_item(item_id)`: return the names of all warehouses that stock a given item, sorted alphabetically. One `SELECT` with a `JOIN`.


In [ ]:
def warehouses_with_item(item_id):
    # TODO: write a SELECT that joins warehouse with stores,
    #       filters on s.item_id = ?, orders by w.name.
    raise NotImplementedError

# Should return [("Berlin",), ("Munich",)] for item 2 (Gadget).
warehouses_with_item(2)


**What we gained:** every constraint is enforced by the engine, the query is one declarative `SELECT`, the data lives in a real file with transactions and indexes.

**What we paid:** every interaction with the DB is a hand-written SQL string. Column lists in the code must stay in sync with `CREATE TABLE`. Results come back as tuples — the structure of the row is implicit. Adding a column is a chore in three places.

## 3 — ORM (SQLAlchemy 2.x)


In [ ]:
%pip install -q "sqlalchemy>=2"


In [ ]:
from pathlib import Path
from sqlalchemy import create_engine, ForeignKey, CheckConstraint, event, select, func
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship, Session

DB_FILE = Path("warehouse_orm.db")
if DB_FILE.exists():
    DB_FILE.unlink()

engine = create_engine(f"sqlite:///{DB_FILE}")

@event.listens_for(engine, "connect")
def _enable_fk(dbapi_con, _):
    dbapi_con.execute("PRAGMA foreign_keys = ON")

class Base(DeclarativeBase):
    pass

class Warehouse(Base):
    __tablename__ = "warehouse"
    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(unique=True)
    address: Mapped[str]
    items: Mapped[list["Item"]] = relationship(
        secondary="stores", back_populates="warehouses"
    )

class Item(Base):
    __tablename__ = "item"
    __table_args__ = (
        CheckConstraint("price  >= 0"),
        CheckConstraint("weight >= 0"),
    )
    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str]
    price: Mapped[float]
    weight: Mapped[float]
    warehouses: Mapped[list[Warehouse]] = relationship(
        secondary="stores", back_populates="items"
    )

class Stores(Base):
    __tablename__ = "stores"
    warehouse_id: Mapped[int] = mapped_column(ForeignKey("warehouse.id"), primary_key=True)
    item_id:      Mapped[int] = mapped_column(ForeignKey("item.id"),      primary_key=True)

Base.metadata.create_all(engine)
session = Session(engine)


In [ ]:
def add_warehouse(name, address):
    w = Warehouse(name=name, address=address)
    session.add(w); session.commit()
    return w

def add_item(name, price, weight):
    i = Item(name=name, price=price, weight=weight)
    session.add(i); session.commit()
    return i

def stock(warehouse_id, item_id):
    session.add(Stores(warehouse_id=warehouse_id, item_id=item_id))
    session.commit()

def items_at(warehouse_id):
    return session.get(Warehouse, warehouse_id).items  # via the relationship

def warehouse_totals():
    stmt = (
        select(Warehouse.name, func.sum(Item.price).label("total"))
        .join(Stores, Stores.warehouse_id == Warehouse.id)
        .join(Item,   Item.id           == Stores.item_id)
        .group_by(Warehouse.id, Warehouse.name)
        .order_by(func.sum(Item.price).desc())
    )
    return session.execute(stmt).all()

add_warehouse("Berlin", "Alexanderplatz 1")
add_warehouse("Munich",  "Marienplatz 8")
add_item("Widget",  9.99, 0.5)
add_item("Gadget",  19.99, 1.2)
add_item("Sprocket", 4.50, 0.1)
stock(1, 1); stock(1, 2); stock(2, 2); stock(2, 3)

# Notice: items_at returns Item objects you can use directly — no manual mapping.
berlin = session.get(Warehouse, 1)
print("items at Berlin:", [(i.name, i.price) for i in berlin.items])
print("warehouse totals:", warehouse_totals())


### The same three buggy inputs


In [ ]:
from sqlalchemy.exc import IntegrityError

for label, action in [
    ("duplicate warehouse name", lambda: add_warehouse("Berlin", "x")),
    ("negative price",            lambda: add_item("Free money", -100, 0)),
    ("FK to missing warehouse",   lambda: stock(999, 1)),
]:
    try:
        action()
        print(f"{label}: NOT BLOCKED")
    except IntegrityError as e:
        session.rollback()
        print(f"{label}: blocked — {type(e).__name__}")


### Your task

The same task as before, but now in SQLAlchemy: return the names of all warehouses that stock a given item, sorted alphabetically.


In [ ]:
def warehouses_with_item(item_id):
    # TODO: build a select(...) that returns Warehouse.name,
    #       joins via Stores, filters on Stores.item_id == item_id,
    #       orders by Warehouse.name. Execute it on `session`.
    raise NotImplementedError

warehouses_with_item(2)


**What changed from DDA:**
- the schema *is* the Python class — no parallel CREATE TABLE to keep in sync
- `items_at` is now `warehouse.items`, an attribute on a real object
- inserts take Python objects, not column-position tuples

**What didn't change:**
- the constraints still come from the database (the ORM tells the DB about them)
- complex aggregations (`warehouse_totals`) still need an explicit `select()` with joins

**What you trade away:**
- a learning curve (relationships, sessions, lazy loading, `flush` vs `commit`)
- some queries are hidden — `warehouse.items` is a SQL query, but you can't see it without turning on logging
- when performance matters, you eventually drop back to raw SQL anyway

## Reflection

Three implementations of the same five operations. Compare:

| | Pure Python | DDA | ORM |
|---|---|---|---|
| Persistence | manual JSON dump | yes (DB file) | yes (DB file) |
| Constraints | none | declared in CREATE TABLE | declared on Python classes |
| Query language | Python loops | SQL strings | Python expressions → SQL |
| Result shape | dicts | tuples | objects |
| Boilerplate to add a column | one place (the dict) | two (CREATE TABLE + queries) | one (the class) |
| Performance ceiling | low | high | high (until you outgrow the abstraction) |

Which would you rather maintain in five years?